# CS6013 sandbox — compress → decompress → upload (Kaggle)

**Runs on a CPU session.** compress/decompress never build the model (they stream
safetensors), so this needs no GPU and does not touch your 30 h/week GPU quota.

### Before running
1. Accelerator: **None (CPU)**
2. Internet: **On**
3. Attach the Qwen3.5-4B base **Model** (mounts read-only under `/kaggle/input`)
4. Add-ons → Secrets: `GITHUB_PAT`, `HF_TOKEN`

### What it does
`git pull sandbox` → `compress.py` → **size gate** → `decompress.py` → verify +
patch → upload to HF (**at repo root**) → commit results back to git.


## 0. CONFIG — the only cell you normally edit

In [ ]:
# Which run folder in the sandbox repo to execute: runs/<RUN_ID>/
RUN_ID = "w01_c40_int4"

# Your PRIVATE sandbox repo, "owner/name"
GITHUB_REPO = "your-github-user/cs6013-sandbox"
GIT_BRANCH = "main"

# HuggingFace
HF_USER = "your-hf-user"
ENROLL = "23B1266"
WEEK = "04"
TARGET = 10          # 10 | 20 | 40  -- the compression target, in percent
SUBMISSION = "01"

# Eval scratch repo (private): the fp16 model molab will serve.
# Files land at the repo ROOT -- vLLM cannot read a subdirectory.
HF_EVAL_REPO = f"{HF_USER}/qwen35-{RUN_ID}-fp16"

# Submission repo (public): the COMPRESSED checkpoint. Spec-mandated name.
HF_SUBMIT_REPO = (
    f"{HF_USER}/{ENROLL}-Week{WEEK}-Compression{TARGET}-Submission{SUBMISSION}"
)

UPLOAD_EVAL = True       # push restored fp16 -> HF_EVAL_REPO
UPLOAD_SUBMISSION = False  # push compressed  -> HF_SUBMIT_REPO (set True when ready)

# Base model. Leave None to auto-discover under /kaggle/input.
BASE_MODEL_PATH = None
MODEL_NAME = "Qwen/Qwen3.5-4B"

print(f"run        {RUN_ID}")
print(f"target     {TARGET}%")
print(f"eval repo  {HF_EVAL_REPO}")
print(f"submit     {HF_SUBMIT_REPO}  (upload={UPLOAD_SUBMISSION})")

## 1. Secrets, paths, and the sandbox repo

In [ ]:
import os, sys, json, shutil, subprocess, pathlib, time

from kaggle_secrets import UserSecretsClient
_sec = UserSecretsClient()
GH_PAT = _sec.get_secret("GITHUB_PAT")
HF_TOKEN = _sec.get_secret("HF_TOKEN")

WORK = pathlib.Path("/kaggle/working/work")
SANDBOX = pathlib.Path("/kaggle/working/sandbox")
WORK.mkdir(parents=True, exist_ok=True)

def sh(cmd, cwd=None, check=True, quiet=False):
    """Run a command, never echoing the PAT."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    shown = " ".join(str(c).replace(GH_PAT, "***") for c in cmd)
    if not quiet:
        print(f"$ {shown}")
        if r.stdout.strip():
            print(r.stdout[-4000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-4000:].replace(GH_PAT, "***"))
        if check:
            raise RuntimeError(f"failed ({r.returncode}): {shown}")
    return r

REMOTE = f"https://{GH_PAT}@github.com/{GITHUB_REPO}.git"

if SANDBOX.exists():
    sh(["git", "fetch", "origin"], cwd=SANDBOX)
    sh(["git", "reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=SANDBOX)
else:
    sh(["git", "clone", "--branch", GIT_BRANCH, REMOTE, str(SANDBOX)])

RUN_DIR = SANDBOX / "runs" / RUN_ID
if not RUN_DIR.is_dir():
    raise SystemExit(
        f"runs/{RUN_ID}/ not found. Available: "
        f"{sorted(p.name for p in (SANDBOX/'runs').glob('*')) if (SANDBOX/'runs').exists() else 'none'}"
    )

for f in ("compress.py", "decompress.py"):
    if not (RUN_DIR / f).is_file():
        raise SystemExit(f"runs/{RUN_ID}/{f} is missing")

print("\nrun dir contents:")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file() and "__pycache__" not in str(p):
        print(f"  {p.relative_to(RUN_DIR)}")

## 2. Optional per-run overrides (`run.yaml`)

```yaml
target: 40
compress_args:   ["--profile", "int4"]
decompress_args: ["--out-dtype", "float16"]
notes: "int4 g128 baseline"
```


In [ ]:
import yaml

RUN_CFG = {}
_rc = RUN_DIR / "run.yaml"
if _rc.is_file():
    RUN_CFG = yaml.safe_load(_rc.read_text()) or {}
    print(yaml.safe_dump(RUN_CFG, sort_keys=False))
else:
    print("no run.yaml — using notebook defaults")

TARGET = int(RUN_CFG.get("target", TARGET))
COMPRESS_ARGS = [str(a) for a in RUN_CFG.get("compress_args", [])]
DECOMPRESS_ARGS = [str(a) for a in RUN_CFG.get("decompress_args", ["--out-dtype", "float16"])]
NOTES = RUN_CFG.get("notes", "")
print(f"\ntarget={TARGET}%  compress_args={COMPRESS_ARGS}  decompress_args={DECOMPRESS_ARGS}")

## 3. Locate the base model and check disk

In [ ]:
def find_base_model():
    hits = []
    root = pathlib.Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            d = cfg.parent
            if list(d.glob("*.safetensors")):
                hits.append(d)
    return sorted(set(hits))

if BASE_MODEL_PATH:
    BASE = pathlib.Path(BASE_MODEL_PATH)
else:
    cands = find_base_model()
    print("candidates under /kaggle/input:")
    for c in cands:
        n = sum(f.stat().st_size for f in c.glob("*.safetensors"))
        print(f"  {n/1024**3:>7.2f} GiB  {c}")
    if not cands:
        raise SystemExit("No model found under /kaggle/input. Attach the base model.")
    qwen = [c for c in cands if "qwen" in str(c).lower()]
    BASE = (qwen or cands)[0]

BASE_BYTES = sum(f.stat().st_size for f in BASE.glob("*.safetensors"))
print(f"\nBASE       {BASE}")
print(f"weights    {BASE_BYTES:,} bytes ({BASE_BYTES/1024**3:.3f} GiB)")
print(f"reference  9,319,737,856 bytes  <- Qwen3.5-4B bf16")
if BASE_BYTES != 9_319_737_856:
    print("NOTE: differs from the reference checkpoint size.")

for f in ("config.json", "chat_template.jinja", "tokenizer.json"):
    print(f"  {'OK     ' if (BASE/f).is_file() else 'MISSING'} {f}")

_t, _u, _f = shutil.disk_usage("/kaggle/working")
print(f"\ndisk /kaggle/working: {_f/1024**3:.1f} GiB free of {_t/1024**3:.1f} GiB")
print(f"need ~{(BASE_BYTES*TARGET/100 + BASE_BYTES)/1024**3:.1f} GiB "
      f"(compressed + restored). Base is read-only in /kaggle/input and does not count.")

## 4. Install run dependencies

In [ ]:
_pp = RUN_DIR / "pyproject.toml"
if _pp.is_file():
    sh([sys.executable, "-m", "pip", "install", "-q", "-e", str(RUN_DIR)], check=False)
else:
    print("no pyproject.toml in the run dir")

# Always present so the pipeline itself can run.
sh([sys.executable, "-m", "pip", "install", "-q",
    "safetensors", "huggingface_hub", "tqdm", "pyyaml"], check=False)
print("deps ready")

## 5. Compress

In [ ]:
COMPRESSED = WORK / RUN_ID / "compressed"
RESTORED = WORK / RUN_ID / "restored_fp16"
if COMPRESSED.exists():
    shutil.rmtree(COMPRESSED)

_t0 = time.time()
sh([sys.executable, "compress.py",
    "--model_name", MODEL_NAME,
    "--checkpoint_path", str(BASE),
    "--output_path", str(COMPRESSED)] + COMPRESS_ARGS,
   cwd=RUN_DIR)
COMPRESS_SEC = time.time() - _t0
print(f"\ncompress took {COMPRESS_SEC:.0f}s")

## 6. Size gate — measured bytes, not predicted

In [ ]:
def dir_bytes(d, pattern="*"):
    return sum(p.stat().st_size for p in pathlib.Path(d).rglob(pattern) if p.is_file())

comp_weights = dir_bytes(COMPRESSED, "*.safetensors")
comp_all = dir_bytes(COMPRESSED)
ratio_weights = comp_weights / BASE_BYTES * 100
ratio_all = comp_all / BASE_BYTES * 100

print(f"base           {BASE_BYTES:,}")
print(f"compressed     {comp_weights:,} weights | {comp_all:,} all files")
print(f"RATIO weights  {ratio_weights:6.2f}%")
print(f"RATIO all      {ratio_all:6.2f}%\n")

for f in sorted(COMPRESSED.rglob("*"), key=lambda p: -p.stat().st_size if p.is_file() else 0):
    if f.is_file():
        print(f"  {f.stat().st_size:>15,}  {f.relative_to(COMPRESSED)}")

# The HF checkpoint must contain ONLY what is needed to load/decompress/eval.
BANNED = ("report", "log", ".ipynb", ".csv", ".py", ".pt", ".bin", "README", "LICENSE")
stray = [str(f.relative_to(COMPRESSED)) for f in COMPRESSED.rglob("*")
         if f.is_file() and any(b.lower() in str(f.name).lower() for b in BANNED)]
print("\ncleanliness:", "OK" if not stray else f"STRAY FILES {stray}")

SIZE_OK = ratio_all <= TARGET
print(f"\n[{'PASS' if SIZE_OK else 'FAIL'}] target {TARGET}%: measured {ratio_all:.2f}%")
if not SIZE_OK:
    print("Over budget — fix the recipe before uploading. Continuing to decompress anyway.")

## 7. Decompress

In [ ]:
if RESTORED.exists():
    shutil.rmtree(RESTORED)

_t0 = time.time()
sh([sys.executable, "decompress.py",
    "--model_name", MODEL_NAME,
    "--checkpoint_path", str(COMPRESSED),
    "--output_path", str(RESTORED)] + DECOMPRESS_ARGS,
   cwd=RUN_DIR)
DECOMPRESS_SEC = time.time() - _t0
print(f"\ndecompress took {DECOMPRESS_SEC:.0f}s")

## 8. Verify and patch the restored checkpoint

Two silent killers, both fixed here:

- **`chat_template.jinja` missing** → `enable_thinking` does nothing, the model
  stops emitting `<think>`, and your eval quietly stops matching the graded
  config with no error.
- **config says `bfloat16` while weights are fp16** → vLLM's `--dtype auto`
  upcasts fp16→bf16 at load (10 mantissa bits → 7), so you lose precision on
  top of quantization and blame the compression.


In [ ]:
REQUIRED = ["config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"]
missing = [f for f in REQUIRED if not (RESTORED / f).is_file()]
for f in REQUIRED:
    print(f"  {'OK     ' if (RESTORED/f).is_file() else 'MISSING'} {f}")

# Copy anything the run's decompress.py forgot, straight from the base.
for f in missing[:]:
    if (BASE / f).is_file():
        shutil.copy2(BASE / f, RESTORED / f)
        print(f"  patched in {f} from base")
        missing.remove(f)
if missing:
    print(f"\nSTILL MISSING {missing} — vLLM may fail or silently drop thinking mode.")

rest_bytes = dir_bytes(RESTORED, "*.safetensors")
print(f"\nrestored weights {rest_bytes:,} bytes ({rest_bytes/1024**3:.3f} GiB)")
print(f"base             {BASE_BYTES:,} bytes")

_cfgp = RESTORED / "config.json"
_cfg = json.loads(_cfgp.read_text())
print("architectures:", _cfg.get("architectures"))

from safetensors import safe_open
_w = sorted(RESTORED.glob("*.safetensors"))[0]
with safe_open(str(_w), framework="pt") as _f:
    _k = next(iter(_f.keys()))
    ACTUAL_DTYPE = _f.get_slice(_k).get_dtype()
print("actual tensor dtype:", ACTUAL_DTYPE)

# Make config agree with the bytes on disk, so --dtype auto is correct for the
# TAs too (they run decompress.py themselves and we do not control their flags).
if ACTUAL_DTYPE in ("F16", "float16"):
    changed = False
    if _cfg.get("dtype") != "float16":
        _cfg["dtype"] = "float16"; changed = True
    if isinstance(_cfg.get("text_config"), dict) and _cfg["text_config"].get("dtype") != "float16":
        _cfg["text_config"]["dtype"] = "float16"; changed = True
    if changed:
        _cfgp.write_text(json.dumps(_cfg, indent=2))
        print("\nPATCHED config.json dtype -> float16")
        print("Move this into your decompress.py so the TAs get it too.")
    else:
        print("\nconfig dtype already float16")

## 9. Upload to HuggingFace — files land at the repo ROOT

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

# vLLM cannot read a subdirectory: config.json must sit at the repo root.
# path_in_repo is deliberately omitted.
if UPLOAD_EVAL:
    api.create_repo(HF_EVAL_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(RESTORED), repo_id=HF_EVAL_REPO,
                      repo_type="model",
                      commit_message=f"{RUN_ID}: restored fp16")
    print("uploaded fp16 ->", HF_EVAL_REPO)
else:
    print("UPLOAD_EVAL is False")

if UPLOAD_SUBMISSION:
    if not SIZE_OK:
        raise SystemExit(f"refusing to upload: {ratio_all:.2f}% > {TARGET}% target")
    if stray:
        raise SystemExit(f"refusing to upload: stray files {stray}")
    api.create_repo(HF_SUBMIT_REPO, repo_type="model", private=False, exist_ok=True)
    api.upload_folder(folder_path=str(COMPRESSED), repo_id=HF_SUBMIT_REPO,
                      repo_type="model",
                      commit_message=f"{RUN_ID}: compressed to {ratio_all:.2f}%")
    print("uploaded compressed ->", HF_SUBMIT_REPO)
else:
    print("UPLOAD_SUBMISSION is False")

## 10. Record results and push them back to git

In [ ]:
RESULTS_DIR = SANDBOX / "results" / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

record = {
    "run_id": RUN_ID,
    "stage": "compress_decompress",
    "utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "base_model": MODEL_NAME,
    "base_bytes": BASE_BYTES,
    "target_pct": TARGET,
    "compressed_weights_bytes": comp_weights,
    "compressed_all_bytes": comp_all,
    "ratio_weights_pct": round(ratio_weights, 3),
    "ratio_all_pct": round(ratio_all, 3),
    "size_gate_pass": bool(SIZE_OK),
    "restored_bytes": rest_bytes,
    "restored_dtype": ACTUAL_DTYPE,
    "stray_files": stray,
    "compress_sec": round(COMPRESS_SEC),
    "decompress_sec": round(DECOMPRESS_SEC),
    "compress_args": COMPRESS_ARGS,
    "decompress_args": DECOMPRESS_ARGS,
    "hf_eval_repo": HF_EVAL_REPO if UPLOAD_EVAL else None,
    "hf_submit_repo": HF_SUBMIT_REPO if UPLOAD_SUBMISSION else None,
    "notes": NOTES,
}
(RESULTS_DIR / "compression.json").write_text(json.dumps(record, indent=2))

# Copy any report the run's own code produced, but never a checkpoint.
for p in (WORK / RUN_ID).glob("*.json"):
    shutil.copy2(p, RESULTS_DIR / p.name)

print(json.dumps(record, indent=2))

sh(["git", "config", "user.email", "kaggle@sandbox.local"], cwd=SANDBOX)
sh(["git", "config", "user.name", "kaggle-sandbox"], cwd=SANDBOX)
sh(["git", "add", "results"], cwd=SANDBOX)
_st = sh(["git", "status", "--porcelain"], cwd=SANDBOX, quiet=True)
if _st.stdout.strip():
    sh(["git", "commit", "-m", f"results({RUN_ID}): compress {ratio_all:.2f}% of base"], cwd=SANDBOX)
    sh(["git", "push", "origin", GIT_BRANCH], cwd=SANDBOX)
    print("\npushed results to git")
else:
    print("\nnothing to commit")

## 11. Cleanup

The restored fp16 checkpoint is ~9.3 GB. Once it is on HF, delete it — you can
always regenerate it. Uncomment to run.


In [ ]:
_t, _u, _f = shutil.disk_usage("/kaggle/working")
print(f"disk free: {_f/1024**3:.1f} GiB")
print(f"compressed {dir_bytes(COMPRESSED)/1024**3:.2f} GiB at {COMPRESSED}")
print(f"restored   {dir_bytes(RESTORED)/1024**3:.2f} GiB at {RESTORED}")

# shutil.rmtree(RESTORED); print("deleted restored")
# shutil.rmtree(COMPRESSED); print("deleted compressed")

print(f"\nNEXT: molab_eval.py with")
print(f'  RUN_ID       = "{RUN_ID}"')
print(f'  HF_EVAL_REPO = "{HF_EVAL_REPO}"')